In [1]:
import os

def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

api_key = None
if in_colab():
    from google.colab import userdata
    api_key = userdata.get("NDIF_API_KEY")
else:
    api_key = os.environ.get("NDIF_API_KEY")

if api_key is None:
    os.environ["NDIF_API_KEY"] = input("Enter NDIF API key: ")

In [2]:
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    !pip install -U nnsight

In [3]:
from IPython.display import clear_output
import einops
import torch
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab" if is_colab else "plotly_mimetype+notebook_connected+notebook"


from nnsight import LanguageModel

c:\Users\Claire Schlesinger\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import nnsight
print(nnsight.__version__)

0.6.3


In [5]:
"""
Uniform Discretized Integrated Gradients using NNsight / NDIF (remote only).

Implements the Integrated Gradients attribution method (Sundararajan et al., 2017)
for language models hosted on NDIF via NNsight remote execution.

Strategy:
  Each IG step runs in a SINGLE trace with a SINGLE invoke:
    1. Access the embedding output (on-device proxy)
    2. Call .requires_grad_(True) on it (standard NNsight gradient pattern)
    3. Scale it in-place by alpha (for zero baseline: interp = alpha * embeds)
    4. Forward pass continues through remaining layers to logits
    5. Backward via `with target_logit.backward():` context
    6. Read .grad on the embedding output

  This avoids cross-invoke complexity entirely and follows the exact
  gradient pattern from the NNsight documentation.

Requirements:
    pip install nnsight torch
"""

from __future__ import annotations

import torch
from typing import Optional, List
from nnsight import LanguageModel
from retry import retry

# ---------------------------------------------------------------------------
# Model-architecture helpers
# ---------------------------------------------------------------------------

_EMBED_PATHS = {
    "gpt2":    "transformer.wte",
    "gpt_neo": "transformer.wte",
    "llama":   "model.embed_tokens",
    "mistral": "model.embed_tokens",
    "gemma":   "model.embed_tokens",
    "gemma2":  "model.embed_tokens",
    "phi":     "model.embed_tokens",
    "phi3":    "model.embed_tokens",
    "qwen2":   "model.embed_tokens",
    "opt":     "model.decoder.embed_tokens",
    "pythia":  "gpt_neox.embed_in",
    "bloom":   "transformer.word_embeddings",
}


def _detect_model_family(model: LanguageModel) -> str:
    model_type = getattr(model.config, "model_type", "").lower()
    for family in _EMBED_PATHS:
        if family in model_type:
            return family
    raise ValueError(
        f"Unknown model type '{model_type}'. Supported: {list(_EMBED_PATHS.keys())}. "
        f"Pass `layer_path` explicitly or add your model to _EMBED_PATHS."
    )


def _get_embedding_path(model: LanguageModel) -> str:
    return _EMBED_PATHS[_detect_model_family(model)]


def _resolve_module(obj, dot_path: str):
    """Navigate 'transformer.h.0' -> obj.transformer.h[0]"""
    current = obj
    for part in dot_path.split("."):
        current = current[int(part)] if part.isdigit() else getattr(current, part)
    return current


# ---------------------------------------------------------------------------
# Core: single-step gradient (remote-safe, single invoke, zero baseline)
#
# Follow the canonical NNsight gradient pattern:
#   1. Access module output as a proxy
#   2. requires_grad_(True) on the proxy
#   3. Modify it (scale by alpha for interpolation)
#   4. Access downstream logits
#   5. with logit.backward(): read .grad
#
# This works because NNsight intercepts the module output, and
# requires_grad_(True) makes it a grad-tracked tensor WITHIN the
# existing computation graph. The model's forward pass continues
# using the modified (scaled) tensor.
# ---------------------------------------------------------------------------

def _compute_ig_step(
    model: LanguageModel,
    input_text: str,
    alpha: float,
    embed_module_path: str,
    target_token_idx: int,
    target_class: int,
) -> torch.Tensor:
    """
    Compute d(target_logit)/d(embeddings) at interpolation point alpha
    (zero baseline: interp = alpha * original_embeddings).
    Single invoke, single trace, entirely on the remote device.
    """

    with model.trace(input_text, remote=True):
        # 1. Access embedding output — this is a real tensor proxy on device
        embed_mod = _resolve_module(model, embed_module_path)
        embeds = embed_mod.output

        # 2. Enable gradient tracking (standard NNsight pattern)
        embeds.requires_grad_(True)

        # 3. Scale embeddings by alpha for interpolation (zero baseline)
        #    Overwrite the module output so downstream layers see the scaled version
        embed_mod.output = embeds * alpha

        # 4. Continue forward to logits
        logits = model.output.logits
        target_logit = logits[0, target_token_idx, target_class]

        # 5. Backward pass — access grad inside the backward context
        with target_logit.backward():
            grad = embeds.grad.save()

    return grad.detach()


def _compute_ig_step_with_baseline(
    model: LanguageModel,
    input_text: str,
    baseline_text: str,
    alpha: float,
    embed_module_path: str,
    target_token_idx: int,
    target_class: int,
) -> torch.Tensor:
    """
    Compute gradient at interpolation point alpha with a text baseline.
    Uses two invokes + barrier: one for baseline embeds, one for
    the interpolated forward+backward.
    """

    with model.trace(remote=True) as tracer:
        barrier = tracer.barrier(2)

        # Invoke 1: capture baseline embeddings
        with tracer.invoke(baseline_text):
            baseline_embeds = _resolve_module(model, embed_module_path).output.clone()
            barrier()

        # Invoke 2: interpolated forward + backward
        with tracer.invoke(input_text):
            barrier()

            embed_mod = _resolve_module(model, embed_module_path)
            embeds = embed_mod.output
            embeds.requires_grad_(True)

            # interp = baseline + alpha * (input - baseline)
            embed_mod.output = baseline_embeds + alpha * (embeds - baseline_embeds)

            logits = model.output.logits
            target_logit = logits[0, target_token_idx, target_class]

            with target_logit.backward():
                grad = embeds.grad.save()

    return grad.detach()


# ---------------------------------------------------------------------------
# Compute embedding delta (input - baseline)
# ---------------------------------------------------------------------------

def _compute_embed_delta(
    model: LanguageModel,
    input_text: str,
    baseline_text: Optional[str],
    embed_module_path: str,
) -> torch.Tensor:
    """Return (input_embeds - baseline_embeds) computed on-device."""

    if baseline_text is None:
        # Zero baseline: delta = input_embeds
        with model.trace(input_text, remote=True):
            delta = _resolve_module(model, embed_module_path).output.clone().save()
        return delta.detach()
    else:
        with model.trace(remote=True) as tracer:
            barrier = tracer.barrier(2)
            with tracer.invoke(input_text):
                inp = _resolve_module(model, embed_module_path).output.clone()
                barrier()
            with tracer.invoke(baseline_text):
                barrier()
                base = _resolve_module(model, embed_module_path).output.clone()
                delta = (inp - base).save()
        return delta.detach()


# ---------------------------------------------------------------------------
# Core: Uniform Discretized Integrated Gradients
# ---------------------------------------------------------------------------

@retry()
def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_idx: int = -1,
    target_class: Optional[int] = None,
    baseline_text: Optional[str] = None,
    m_steps: int = 50,
) -> torch.Tensor:
    """
    Compute Uniform Discretized Integrated Gradients at the embedding level.

    Parameters
    ----------
    model : LanguageModel
        An nnsight LanguageModel (will be run remotely on NDIF).
    input_text : str
        The input prompt to attribute.
    target_token_idx : int
        Which token position's logits to attribute (default: -1, last token).
    target_class : int | None
        Vocabulary index of the target class. If None, uses argmax.
    baseline_text : str | None
        Text for the baseline. If None, uses a zero-embedding baseline.
    m_steps : int
        Number of interpolation steps (midpoint Riemann sum).

    Returns
    -------
    attributions : torch.Tensor
        Shape (seq_len, hidden_dim). Embedding-level attributions.
    """
    embed_module_path = _get_embedding_path(model)
    use_text_baseline = (baseline_text is not None)

    # ------------------------------------------------------------------
    # 1. Determine the target class if not given
    # ------------------------------------------------------------------
    if target_class is None:
        with model.trace(input_text, remote=True):
            logits_out = model.output.logits.save()
        target_class = logits_out[0, target_token_idx].argmax(dim=-1).item()

    # ------------------------------------------------------------------
    # 2. Accumulate gradients at uniform midpoints
    # ------------------------------------------------------------------
    grad_sum = None
    for k in range(m_steps):
        alpha = (k + 0.5) / m_steps

        if use_text_baseline:
            grad = _compute_ig_step_with_baseline(
                model=model,
                input_text=input_text,
                baseline_text=baseline_text,
                alpha=alpha,
                embed_module_path=embed_module_path,
                target_token_idx=target_token_idx,
                target_class=target_class,
            )
        else:
            grad = _compute_ig_step(
                model=model,
                input_text=input_text,
                alpha=alpha,
                embed_module_path=embed_module_path,
                target_token_idx=target_token_idx,
                target_class=target_class,
            )

        if grad_sum is None:
            grad_sum = grad.clone()
        else:
            grad_sum = grad_sum + grad

    avg_grads = grad_sum / m_steps

    # ------------------------------------------------------------------
    # 3. Compute embedding delta (input - baseline)
    # ------------------------------------------------------------------
    delta = _compute_embed_delta(
        model=model,
        input_text=input_text,
        baseline_text=baseline_text,
        embed_module_path=embed_module_path,
    )

    # ------------------------------------------------------------------
    # 4. IG = delta * avg_gradients
    # ------------------------------------------------------------------
    attributions = delta.squeeze(0) * avg_grads.squeeze(0)
    return attributions


# ---------------------------------------------------------------------------
# Higher-level convenience functions
# ---------------------------------------------------------------------------

def token_attributions(
    model: LanguageModel,
    input_text: str,
    target_token_idx: int = -1,
    target_class: Optional[int] = None,
    m_steps: int = 50,
) -> list[tuple[str, float]]:
    """
    Compute per-token attribution scores using IG on the embedding layer.
    Returns a list of (token_string, attribution_score) tuples.
    """
    attrs = integrated_gradients(
        model=model,
        input_text=input_text,
        target_token_idx=target_token_idx,
        target_class=target_class,
        m_steps=m_steps,
    )

    # Sum over embedding dim -> per-token scores
    token_scores = attrs.sum(dim=-1)
    token_ids = model.tokenizer.encode(input_text)
    tokens = [model.tokenizer.decode([tid]) for tid in token_ids]

    return list(zip(tokens, token_scores.tolist()))


def layer_attributions(
    model: LanguageModel,
    input_text: str,
    layer_path: str,
    target_token_idx: int = -1,
    target_class: Optional[int] = None,
    m_steps: int = 50,
) -> torch.Tensor:
    """
    Compute per-neuron attribution at a specific layer.

    Uses the standard NNsight gradient pattern with retain_grad()
    on the target layer's output to capture gradients flowing through it.
    """
    embed_module_path = _get_embedding_path(model)

    if target_class is None:
        with model.trace(input_text, remote=True):
            logits_out = model.output.logits.save()
        target_class = logits_out[0, target_token_idx].argmax(dim=-1).item()

    # Accumulate gradients w.r.t. the target layer's output
    grad_sum = None
    for k in range(m_steps):
        alpha = (k + 0.5) / m_steps

        with model.trace(input_text, remote=True):
            # Scale embeddings for interpolation
            embed_mod = _resolve_module(model, embed_module_path)
            embeds = embed_mod.output
            embeds.requires_grad_(True)
            embed_mod.output = embeds * alpha

            # Access target layer output and retain its gradient
            layer_out = _resolve_module(model, layer_path).output
            if isinstance(layer_out, tuple):
                layer_out = layer_out[0]
            layer_out.retain_grad()

            # Forward to logits
            logits = model.output.logits
            target_logit = logits[0, target_token_idx, target_class]

            # Backward — read grad on the layer output
            with target_logit.backward():
                grad = layer_out.grad.save()

        grad = grad.detach()
        if grad_sum is None:
            grad_sum = grad.clone()
        else:
            grad_sum = grad_sum + grad

    avg_grads = grad_sum / m_steps

    # Compute layer activation delta: act(input) - act(zero_baseline)
    with model.trace(remote=True) as tracer:
        barrier = tracer.barrier(2)

        with tracer.invoke(input_text):
            act_out = _resolve_module(model, layer_path).output
            if isinstance(act_out, tuple):
                act_out = act_out[0]
            act_in = act_out.clone()
            barrier()

        with tracer.invoke(input_text):
            barrier()
            embed_mod = _resolve_module(model, embed_module_path)
            embed_mod.output = embed_mod.output * 0
            act_out2 = _resolve_module(model, layer_path).output
            if isinstance(act_out2, tuple):
                act_out2 = act_out2[0]
            delta = (act_in - act_out2).save()

    delta = delta.detach()
    attributions = delta.squeeze(0) * avg_grads.squeeze(0)
    return attributions


# ---------------------------------------------------------------------------
# Convergence check
# ---------------------------------------------------------------------------

def check_convergence(
    model: LanguageModel,
    input_text: str,
    target_token_idx: int = -1,
    target_class: Optional[int] = None,
    step_counts: Optional[List[int]] = None,
) -> list[tuple[int, float]]:
    """
    Evaluate completeness: sum(IG) ≈ F(x) - F(baseline).
    Returns list of (m_steps, approximation_error).
    """
    if step_counts is None:
        step_counts = [10, 25, 50, 100, 200]

    embed_module_path = _get_embedding_path(model)

    # F(input)
    with model.trace(input_text, remote=True):
        logits_input = model.output.logits.save()

    if target_class is None:
        target_class = logits_input[0, target_token_idx].argmax().item()
    f_input = logits_input[0, target_token_idx, target_class].item()

    # F(zero baseline)
    with model.trace(input_text, remote=True):
        embed_mod = _resolve_module(model, embed_module_path)
        embed_mod.output = embed_mod.output * 0
        logits_baseline = model.output.logits.save()

    f_baseline = logits_baseline[0, target_token_idx, target_class].item()
    true_diff = f_input - f_baseline

    results = []
    for m in step_counts:
        attrs = integrated_gradients(
            model=model,
            input_text=input_text,
            target_token_idx=target_token_idx,
            target_class=target_class,
            m_steps=m,
        )
        attr_sum = attrs.sum().item()
        error = abs(attr_sum - true_diff)
        results.append((m, error))
        print(f"  m={m:>4d} | sum(IG)={attr_sum:.4f} | "
              f"F(x)-F(x')={true_diff:.4f} | error={error:.4f}")

    return results

In [6]:
DEF_BLOCK = (
    "DEFINITION OF UNCERTAINTY (Second-Moment):\n\n"

    "Uncertainty measures the VARIANCE or SPREAD of possible outcomes, "
    "not the expected value of outcomes.\n\n"

    "EXAMPLES:\n"
    "UNCERTAINTY (variance):\n"
    "- 'Revenue could be anywhere from $50M to $200M' → wide range\n"
    "- 'It depends on whether the regulation passes' → binary outcomes far apart\n"
    "- 'Roll two dice' → 11 possible sums with different probabilities\n\n"

    "NO UNCERTAINTY (zero variance):\n"
    "- 'Revenue will be $100M' → single outcome\n"
    "- 'We will lose $50M due to the tariff' → bad but certain\n"
    "- '2 + 2 = 4' → deterministic\n\n"

    "KEY PRINCIPLE NOT ABOUT SENTIMENT:\n"
    "- 'We expect difficult market conditions' → negative sentiment, but if the difficulty is certain, this is LOW uncertainty\n"
    "- 'Sales will definitely drop 20%' → bad news but NO uncertainty\n"
)

prompt_lead = DEF_BLOCK + "\n\nRespond only one word: Yes or no depending on if the phrase meets the definition of uncertainty:\n\nPhrase: We will definitely see a fall in our stock price in Q3.\n\nResponse: no\n\nPhrase: We will see some effect to our stock price by our new product.\n\nResponse: yes\n\nPhrase: "
prompt_end = "\n\nResponse:"


In [7]:
import json
with open('../synthetic_data/API/synthetic_pairs_200.json') as f:
    examples = json.load(f)

In [8]:
no_uncertainty = [e["no"] for e in examples]
yes_uncertainty = [e["high"] for e in examples]

In [9]:
no_uncertainty_prompts = [prompt_lead + e + prompt_end for e in no_uncertainty]
yes_uncertainty_prompts = [prompt_lead + e + prompt_end for e in yes_uncertainty]

In [10]:
"""
Word-level N-gram Attribution Analysis using Integrated Gradients on NDIF.

Given a list of prompts, computes embedding-level IG attributions, maps
subword token scores back to whole words, then extracts and ranks
word-level unigrams, bigrams, and trigrams.

Requires the integrated_gradients module from the previous artifact
(assumed to be saved as `ig_nnsight.py` or pasted into the same notebook).

Usage:
    python ngram_attribution.py
"""

from __future__ import annotations

import re
import torch
from typing import Optional
from collections import defaultdict
from nnsight import LanguageModel

# ---------------------------------------------------------------------------
# Subword → word mapping
# ---------------------------------------------------------------------------

def _merge_subwords_to_words(
    model: LanguageModel,
    input_text: str,
    token_scores: torch.Tensor,
) -> list[tuple[str, float]]:
    """
    Map subword token scores back to whole words.

    Strategy:
      1. Encode the input to get token IDs.
      2. Decode each token to its string form.
      3. Walk through tokens: if a token starts with a space or is the
         first token, it begins a new word. Otherwise it continues the
         current word (subword continuation).
      4. Each word's score is the sum of its subword token scores.
      5. The word's text is the concatenation of its subword strings,
         stripped of extra whitespace.

    This handles BPE (GPT-2, LLaMA, Mistral, etc.) where word-initial
    tokens have a leading space and continuations do not.
    """
    token_ids = model.tokenizer.encode(input_text)
    token_strs = [model.tokenizer.decode([tid]) for tid in token_ids]

    words = []       # list of (word_text, word_score)
    current_text = ""
    current_score = 0.0

    for i, (tok_str, score) in enumerate(zip(token_strs, token_scores.tolist())):
        # Detect word boundary: leading space, or first token, or punctuation-only token
        is_new_word = (
            i == 0
            or tok_str.startswith(" ")
            or tok_str.startswith("\n")
            # Handle tokenizers that use Ġ (GPT-2 byte-level BPE)
            or tok_str.startswith("Ġ")
            # Standalone punctuation starts a new "word"
            or (len(tok_str.strip()) > 0 and tok_str.strip()[0] in ".,;:!?()[]{}\"'`~@#$%^&*-+=/<>|\\")
        )

        if is_new_word and i > 0:
            # Flush the previous word
            word = current_text.strip()
            if word:
                words.append((word, current_score))
            current_text = tok_str
            current_score = score
        else:
            current_text += tok_str
            current_score += score

    # Flush last word
    word = current_text.strip()
    if word:
        words.append((word, current_score))

    return words


# ---------------------------------------------------------------------------
# Per-prompt: get word-level scores
# ---------------------------------------------------------------------------

def _get_word_scores(
    model: LanguageModel,
    input_text: str,
    target_token_idx: int = -1,
    target_class: Optional[int] = None,
    m_steps: int = 30,
) -> list[tuple[str, float]]:
    """
    Run IG on a single prompt and return word-level (word_str, score) pairs.
    """
    attrs = integrated_gradients(
        model=model,
        input_text=input_text,
        target_token_idx=target_token_idx,
        target_class=target_class,
        m_steps=m_steps,
    )

    # Sum across embedding dim → per-token scores
    token_scores = attrs.sum(dim=-1)  # (seq_len,)

    # Merge subwords into words
    return _merge_subwords_to_words(model, input_text, token_scores)


# ---------------------------------------------------------------------------
# N-gram extraction from word-score lists
# ---------------------------------------------------------------------------

def _extract_word_ngrams(
    word_scores: list[tuple[str, float]],
    n: int,
) -> list[tuple[str, float]]:
    """
    Extract word-level n-grams. Each n-gram's score is the sum of its
    constituent word scores. The n-gram text joins words with spaces.
    """
    ngrams = []
    for i in range(len(word_scores) - n + 1):
        window = word_scores[i : i + n]
        text = " ".join(w for w, _ in window)
        score = sum(s for _, s in window)
        ngrams.append((text, score))
    return ngrams


# ---------------------------------------------------------------------------
# Aggregate n-grams across prompts
# ---------------------------------------------------------------------------

def _aggregate_ngrams(
    all_ngrams: list[list[tuple[str, float]]],
    mode: str = "mean",
) -> dict[str, float]:
    """
    Aggregate n-gram scores across prompts.

    Modes:
      "sum"   — total score across all occurrences
      "mean"  — average score per occurrence
      "max"   — maximum score across occurrences (by absolute value)
    """
    accum = defaultdict(list)
    for prompt_ngrams in all_ngrams:
        for text, score in prompt_ngrams:
            key = _normalize(text)
            if key:
                accum[key].append(score)

    result = {}
    for key, scores in accum.items():
        if mode == "sum":
            result[key] = sum(scores)
        elif mode == "mean":
            result[key] = sum(scores) / len(scores)
        elif mode == "max":
            result[key] = max(scores, key=abs)
        else:
            raise ValueError(f"Unknown mode '{mode}'. Use sum/mean/max.")

    return result


def _normalize(text: str) -> str:
    """Lowercase and collapse whitespace for aggregation keys."""
    return re.sub(r"\s+", " ", text.lower()).strip()


# ---------------------------------------------------------------------------
# Main: top-k word-level n-gram attribution
# ---------------------------------------------------------------------------

def topk_ngram_attributions(
    model: LanguageModel,
    prompts: list[str],
    k: int = 10,
    m_steps: int = 30,
    target_token_idx: int = -1,
    target_classes: Optional[list[int]] = None,
    aggregation: str = "mean",
    verbose: bool = True,
) -> dict[str, list[tuple[str, float]]]:
    """
    Compute IG attributions for a list of prompts and return the top-k
    word-level unigrams, bigrams, and trigrams.

    Parameters
    ----------
    model : LanguageModel
        NNsight LanguageModel (runs remotely on NDIF).
    prompts : list[str]
        Input prompts to attribute.
    k : int
        Number of top n-grams to return for each n.
    m_steps : int
        Number of IG interpolation steps per prompt.
    target_token_idx : int
        Token position to attribute (default: -1, last token).
    target_classes : list[int] | None
        Per-prompt target class indices. If None, uses argmax for each.
    aggregation : str
        How to aggregate: "sum", "mean", or "max".
    verbose : bool
        Print progress and results.

    Returns
    -------
    results : dict with keys "unigrams", "bigrams", "trigrams"
        Each is a list of (ngram_text, score), sorted by descending
        absolute score.
    """

    all_word_scores = _collect_word_scores(
        model, prompts, m_steps, target_token_idx, target_classes, verbose
    )

    all_uni = [_extract_word_ngrams(ws, 1) for ws in all_word_scores]
    all_bi = [_extract_word_ngrams(ws, 2) for ws in all_word_scores]
    all_tri = [_extract_word_ngrams(ws, 3) for ws in all_word_scores]

    uni_agg = _aggregate_ngrams(all_uni, mode=aggregation)
    bi_agg = _aggregate_ngrams(all_bi, mode=aggregation)
    tri_agg = _aggregate_ngrams(all_tri, mode=aggregation)

    def topk_sorted(d, k):
        return sorted(d.items(), key=lambda x: abs(x[1]), reverse=True)[:k]

    results = {
        "unigrams": topk_sorted(uni_agg, k),
        "bigrams": topk_sorted(bi_agg, k),
        "trigrams": topk_sorted(tri_agg, k),
    }

    if verbose:
        _print_results(results)

    return results


def topk_ngram_attributions_signed(
    model: LanguageModel,
    prompts: list[str],
    k: int = 10,
    m_steps: int = 30,
    target_token_idx: int = -1,
    target_classes: Optional[list[int]] = None,
    aggregation: str = "mean",
    verbose: bool = True,
) -> dict[str, dict[str, list[tuple[str, float]]]]:
    """
    Like topk_ngram_attributions but splits into positive (promoting)
    and negative (suppressing) n-grams.

    Returns
    -------
    results : dict with keys "unigrams", "bigrams", "trigrams"
        Each is a dict with "positive" and "negative" lists.
    """

    all_word_scores = _collect_word_scores(
        model, prompts, m_steps, target_token_idx, target_classes, verbose
    )

    all_uni = [_extract_word_ngrams(ws, 1) for ws in all_word_scores]
    all_bi = [_extract_word_ngrams(ws, 2) for ws in all_word_scores]
    all_tri = [_extract_word_ngrams(ws, 3) for ws in all_word_scores]

    uni_agg = _aggregate_ngrams(all_uni, mode=aggregation)
    bi_agg = _aggregate_ngrams(all_bi, mode=aggregation)
    tri_agg = _aggregate_ngrams(all_tri, mode=aggregation)

    def split_topk(d, k):
        pos = sorted(
            [(t, s) for t, s in d.items() if s > 0],
            key=lambda x: x[1], reverse=True
        )[:k]
        neg = sorted(
            [(t, s) for t, s in d.items() if s < 0],
            key=lambda x: x[1]
        )[:k]
        return {"positive": pos, "negative": neg}

    results = {
        "unigrams": split_topk(uni_agg, k),
        "bigrams": split_topk(bi_agg, k),
        "trigrams": split_topk(tri_agg, k),
    }

    if verbose:
        _print_signed_results(results, k)

    return results


# ---------------------------------------------------------------------------
# Shared helper: collect word scores for all prompts
# ---------------------------------------------------------------------------

def _collect_word_scores(
    model, prompts, m_steps, target_token_idx, target_classes, verbose
) -> list[list[tuple[str, float]]]:
    all_word_scores = []
    for i, prompt in enumerate(prompts):
        if verbose:
            print(f"  [{i+1}/{len(prompts)}] {prompt[:70]}...")

        tc = target_classes[i] if target_classes is not None else None
        ws = _get_word_scores(
            model=model,
            input_text=prompt,
            target_token_idx=target_token_idx,
            target_class=tc,
            m_steps=m_steps,
        )

        if verbose:
            # Show per-word breakdown for this prompt
            print(f"           Words: {' | '.join(f'{w}({s:.2f})' for w, s in ws)}")

        all_word_scores.append(ws)
    return all_word_scores


# ---------------------------------------------------------------------------
# Pretty-printing
# ---------------------------------------------------------------------------

def _print_results(results: dict[str, list[tuple[str, float]]]):
    for ngram_type in ["unigrams", "bigrams", "trigrams"]:
        items = results[ngram_type]
        print(f"\n{'='*55}")
        print(f"  Top {len(items)} {ngram_type.upper()}")
        print(f"{'='*55}")
        for rank, (text, score) in enumerate(items, 1):
            sign = "+" if score > 0 else "-"
            bar = sign * min(int(abs(score)), 40)
            print(f"  {rank:>3}. {text:>35s}  {score:>10.4f}  {bar}")


def _print_signed_results(results, k):
    for ngram_type in ["unigrams", "bigrams", "trigrams"]:
        data = results[ngram_type]
        print(f"\n{'='*55}")
        print(f"  Top {k} {ngram_type.upper()}")
        print(f"{'='*55}")
        print(f"  {'--- Positive (promoting prediction) ---':>50}")
        for rank, (text, score) in enumerate(data["positive"], 1):
            print(f"  {rank:>3}. {text:>35s}  {score:>+10.4f}")
        print(f"  {'--- Negative (suppressing prediction) ---':>50}")
        for rank, (text, score) in enumerate(data["negative"], 1):
            print(f"  {rank:>3}. {text:>35s}  {score:>+10.4f}")


In [11]:
model = LanguageModel("meta-llama/Llama-3.1-8B")
yes_token_num, no_token_num = model.tokenizer.convert_tokens_to_ids(["yes", "no"])
print(yes_token_num, no_token_num)

9891 2201


In [ ]:
results_yes = topk_ngram_attributions(
    model=model,
    prompts=yes_uncertainty_prompts,
    k=25,
    m_steps=20,
    aggregation="mean",
    target_classes=[yes_token_num] * len(yes_uncertainty_prompts)
)

  [1/200] DEFINITION OF UNCERTAINTY (Second-Moment):

Uncertainty measures the V...


⬇ Downloading: 100%|██████████| 2.12M/2.12M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.07M/2.07M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.09M/2.09M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.05M/2.05M [00:00<00:00]


...

⬇ Downloading: 100%|██████████| 2.00M/2.00M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.00M/2.00M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.03M/2.03M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.03M/2.03M [00:00<00:00]


⬇ Downloading: 100%|██████████| 2.02M/2.02M [00:00<00:00]


In [ ]:
results_no = topk_ngram_attributions(
    model=model,
    prompts=no_uncertainty_prompts,
    k=100,
    m_steps=20,
    aggregation="mean",
    target_token_idx=-1,
    target_classes=[no_token_num] * len(no_uncertainty_prompts)
)

In [ ]:
import json
with open("top_k_importance_results_yes.json", "wt+") as f:
    json.dump(results_yes, f, indent=2)

with open("top_k_importance_results_no.json", "wt+") as f:
    json.dump(results_no, f, indent=2)

In [ ]:
for k, v in results_yes.items():
    v.sort(key=lambda x: x[1])
    print(v)

In [ ]:
for k, v in results_no.items():
    v.sort(key=lambda x: x[1])
    print(v)